# Supp Table 4b: preranked GSEA of the TIL zones

**Produces Supp Table 4b** (`4b_gsea_all_comparisons_combined`): gseapy prerank results
(Hallmark, BioCarta, KEGG Medicus, Reactome, ImmuneSigDB; 1,000 permutations, seed 0) for
- TIL-high vs TIL-low sections within TIL+, BZ1 and BZ2 (ranking: Welch t of the per-section
  mean log-normalised expression of the zone; 57 TIL-high vs 148 TIL-low sections, the 23
  TIL-low sections without TIL+ spots have no TIL+ / border spots);
- TIL+ vs BZ1 and BZ1 vs BZ2 within TIL-high and within TIL-low sections (ranking: paired t
  over the sections that have both zones).

Fig 5f–g (`fig5fg_TIL_zone_GSEA.ipynb`) plot this table.

**Origin**: Spencer Watson, EFS `Analysis_Repo/Notebooks/TIL_Niche_Comparisons_v2_clean.ipynb`,
cells 60 (gene-set library) and 61 (pseudobulk, rankings, prerank), run on 2026-02-25. Cell 61
took the per-section means from the whole log-normalised Visium matrix in memory
(`AllDataCenters.load_spatial_adata`, > 100 GB); here the same means come from
`neighborhoods.pseudobulk_sums`, which reads the matrix a few sections at a time. The loader of
that time also returned the spot image coordinates `imagerow` and `imagecol` as two of the
18,084 "genes" of the matrix, so they were part of the ranked lists (they are in no gene set,
but they count in the list length). `COORDS_IN_RANKING = True` keeps them, as the table was made.

**Compared with the published table** (last cells): same 47,957 rows and columns; ES, leading-edge
genes, Gene % and Tag % identical; NES, p-values and FDR identical (< 1e-14) in six of the seven
comparisons; in TIL-high vs TIL-low within TIL+ they differ by at most 7e-4 (NES) and 2e-4 (FDR), from
the arbitrary order of genes with tied scores. The same 21,442 tests have FDR < 0.05.
The gene space is that of all 228 sections (`section_ids=visium.sections()`): one gene (HNRNPCL3) is
only measured in sections without TIL+ spots, and leaving it out of the ranked lists changes every ES.

In [ ]:
import time

import gseapy as gp
import numpy as np
import pandas as pd

from gbmcollaboration.paper import config, neighborhoods, output, til, visium

In [ ]:
OUT = output.setup("supptables_4b_gsea_TIL_zones")
PUBLISHED = config.EFS / "supplementary_tables/5b_gsea_all_comparisons_combined.csv"
COORDS_IN_RANKING = True     # imagerow / imagecol in the ranked lists, as in the published run

GENE_SET_BASE = config.EFS / "Analysis_Repo/Gene_Sets"

# Add/remove GMT files as available
GMT_PATHS = [
    GENE_SET_BASE / "h.all.v2024.1.Hs.symbols.gmt",              # HALLMARK
    GENE_SET_BASE / "c2.cp.biocarta.v2024.1.Hs.symbols.gmt",     # BIOCARTA
    GENE_SET_BASE / "c2.cp.kegg_medicus.v2024.1.Hs.symbols.gmt", # KEGG MEDICUS
    GENE_SET_BASE / "c2.cp.reactome.v2024.1.Hs.symbols.gmt",     # REACTOME
    GENE_SET_BASE / "c2.cp.pid.v2024.1.Hs.symbols.gmt",          # PID (NCI)
    GENE_SET_BASE / "c5.go.bp.v2024.1.Hs.symbols.gmt",           # GO Biological Process
    GENE_SET_BASE / "c7.immunesigdb.v2024.1.Hs.symbols.gmt",     # ImmuneSigDB
]

# ===== CONFIG =====
ZONE_COL     = "TIL_zone"
GROUP_COL    = "TIL_group"       # "TIL_high" / "TIL_low"
SAMPLE_COL   = "sample"

MIN_SIZE     = 10
MAX_SIZE     = 5000
N_PERM       = 1000
N_PROCESSES  = 4

# Zones to include
ZONES = ["TIL+", "BZ1", "BZ2"]

# Between-group comparisons: TIL_high vs TIL_low within each zone
BETWEEN_GROUP = True

# Between-zone comparisons: adjacent zone pairs within each TIL group
BETWEEN_ZONE = True
ZONE_PAIRS = [
    ("TIL+", "BZ1"),   # positive NES = enriched in TIL+ relative to BZ1
    ("BZ1", "BZ2"),    # positive NES = enriched in BZ1 relative to BZ2
]
GROUPS_FOR_ZONE_COMPARISON = ["TIL_high", "TIL_low"]

# Minimum variance floor to avoid division by zero / infinite t-stats
VAR_FLOOR = 1e-8
# ==================

In [ ]:
def read_gmt_to_dict(path):
    gs = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            name = parts[0]
            genes = [g for g in parts[2:] if g]
            if not genes:
                continue
            if name in gs:
                gs[name].extend(genes)
            else:
                gs[name] = list(genes)
    return {k: sorted(set(v)) for k, v in gs.items()}


# Helper: GSEA column normalization
def _find_col(df, possible_names, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for name in possible_names:
        key = name.lower()
        if key in lower_map:
            return lower_map[key]
    for name in possible_names:
        key = name.lower()
        for low, orig in lower_map.items():
            if key in low:
                return orig
    if required:
        raise KeyError(f"None of {possible_names} found in columns: "
                       f"{list(df.columns)}")
    return None


def normalize_gsea_cols(res2d):
    df = res2d.copy()
    for target, candidates in [
        ("NES", ["nes"]),
        ("FDR", ["fdr q-val", "fdr", "fdr_q", "fdr_qval", "fdr_bh"]),
        ("PVAL", ["nom p-val", "p-val", "pval", "p_value"]),
        ("gene_set", ["term", "name", "gene_set", "gene_set_name"]),
    ]:
        col = _find_col(df, candidates, required=True)
        if col != target:
            df = df.rename(columns={col: target})
    return df


def pseudobulk(zone, group):
    """Return DataFrame: samples x genes (mean expression within the zone's spots of the group)."""
    rows = PB_MEAN.loc[(PB_MEAN.index.get_level_values("label") == zone)
                       & np.asarray(PB_GROUP == group)]
    return rows.droplevel("label")


# ── Ranking functions ────────────────────────────────────────────────

def welch_t_rank(pb_a, pb_b):
    """
    Welch's t-statistic per gene: unpaired, unequal variance.

    t = (mean_A - mean_B) / sqrt(var_A/n_A + var_B/n_B)

    Positive t → higher in group A.
    """
    n_a = pb_a.shape[0]
    n_b = pb_b.shape[0]
    mean_a = pb_a.mean(axis=0)
    mean_b = pb_b.mean(axis=0)
    var_a = pb_a.var(axis=0, ddof=1)
    var_b = pb_b.var(axis=0, ddof=1)

    # Floor variance to avoid inf
    var_a = np.maximum(var_a, VAR_FLOOR)
    var_b = np.maximum(var_b, VAR_FLOOR)

    se = np.sqrt(var_a / n_a + var_b / n_b)
    t = (mean_a - mean_b) / se

    # Replace any remaining inf/nan with 0
    t = t.replace([np.inf, -np.inf], 0).fillna(0)

    return t


def paired_t_rank(pb_a, pb_b, shared_samples):
    """
    Paired t-statistic per gene.

    t = mean(diff) / (std(diff) / sqrt(n))

    where diff = A - B for each paired sample.
    Positive t → higher in zone A.
    """
    diff = pb_a.loc[shared_samples] - pb_b.loc[shared_samples]
    n = len(shared_samples)
    mean_diff = diff.mean(axis=0)
    std_diff = diff.std(axis=0, ddof=1)

    # Floor std to avoid inf
    std_diff = np.maximum(std_diff, np.sqrt(VAR_FLOOR))

    t = mean_diff / (std_diff / np.sqrt(n))

    # Replace any remaining inf/nan with 0
    t = t.replace([np.inf, -np.inf], 0).fillna(0)

    return t


# Helper: run prerank GSEA on a ranking Series
def run_gsea(rnk_series, label):
    """Run gseapy.prerank with combined_sets. Returns normalized res2d."""
    rnk_sorted = rnk_series.sort_values(ascending=False)
    rnk_df = pd.DataFrame({
        "gene": rnk_sorted.index,
        "score": rnk_sorted.values,
    })

    pre_res = gp.prerank(
        rnk=rnk_df,
        gene_sets=combined_sets,
        threads=N_PROCESSES,
        permutation_num=N_PERM,
        min_size=MIN_SIZE,
        max_size=MAX_SIZE,
        seed=0,
        outdir=None,
        format="png",
        verbose=False,
    )

    res = normalize_gsea_cols(pre_res.res2d)
    res["comparison"] = label
    return res


def print_top(res, pos_label, neg_label):
    sig = res[res["FDR"].astype(float) < 0.05]
    for label, sel in [(pos_label, sig["NES"].astype(float) > 0), (neg_label, sig["NES"].astype(float) < 0)]:
        print(f"  Enriched in {label} (FDR<0.05): {int(sel.sum())}")
        top = sig[sel].sort_values("FDR").head(5)
        if not top.empty:
            print(top[["gene_set", "NES", "FDR"]].to_string(index=False))

## Gene-set library

In [ ]:
combined_sets = {}
for gmt_path in GMT_PATHS:
    if not gmt_path.is_file():
        print(f"[SKIP] Not found: {gmt_path.name}")
        continue
    this_lib = read_gmt_to_dict(gmt_path)
    print(f"{gmt_path.name}: {len(this_lib)} gene sets")
    for name, genes in this_lib.items():
        if name in combined_sets:
            combined_sets[name].update(genes)
        else:
            combined_sets[name] = set(genes)

combined_sets = {k: sorted(v) for k, v in combined_sets.items()}
print(f"\nCombined library: {len(combined_sets)} gene sets")

## Per-section zone pseudobulk (mean log-normalised expression)

In [ ]:
t0 = time.time()
zones = til.til_zones()
groups = til.til_groups()["TIL_group"]
labels = zones[ZONE_COL].where(zones[ZONE_COL].isin(ZONES))
# gene space of all 228 sections
sums, n_spots = neighborhoods.pseudobulk_sums(labels.dropna(), normalization="log_norm_count",
                                              section_ids=visium.sections(), n_jobs=4)
if COORDS_IN_RANKING:
    spots, _ = neighborhoods.load_spots()
    coords = spots[["imagerow", "imagecol"]].join(labels.rename("label")).join(spots["sample"].rename("sec"))
    coord_sums = coords.dropna(subset=["label"]).groupby(["sec", "label"])[["imagerow", "imagecol"]].sum()
    sums = sums.join(coord_sums.rename_axis(["sample", "label"]))
PB_MEAN = sums.div(n_spots, axis=0)
PB_GROUP = PB_MEAN.index.get_level_values("sample").map(groups)
print(f"{PB_MEAN.shape[0]} section-zones x {PB_MEAN.shape[1]} genes; {time.time() - t0:.0f} s")
print(pd.crosstab(PB_MEAN.index.get_level_values("label"), PB_GROUP))

## Between-group: TIL-high vs TIL-low within each zone (Welch t)

In [ ]:
all_results = []
t0 = time.time()

if BETWEEN_GROUP:
    for zone in ZONES:
        pb_high = pseudobulk(zone, "TIL_high")
        pb_low = pseudobulk(zone, "TIL_low")

        n_high = pb_high.shape[0]
        n_low = pb_low.shape[0]

        if n_high < 3 or n_low < 3:
            print(f"  [SKIP] {zone}: TIL-High n={n_high}, TIL-Low n={n_low}")
            continue

        # Welch's t-statistic ranking
        rnk = welch_t_rank(pb_high, pb_low)

        label = f"TIL-High_vs_TIL-Low_in_{zone}"
        print(f"\n--- {label} (n_high={n_high}, n_low={n_low}, "
              f"ranking=Welch t-stat) ---")
        print(f"  Ranking stats: median={rnk.median():.3f}, "
              f"mean={rnk.mean():.3f}, "
              f"min={rnk.min():.3f}, max={rnk.max():.3f}, "
              f"n_zero={(rnk == 0).sum()}")

        res = run_gsea(rnk, label)
        res["zone"] = zone
        res["group_A"] = "TIL-High"
        res["group_B"] = "TIL-Low"
        res["comparison_type"] = "between_group"
        res["ranking_metric"] = "welch_t"
        print_top(res, "TIL-High", "TIL-Low")
        all_results.append(res)

## Between-zone: adjacent zones within each TIL group (paired t)

In [ ]:
if BETWEEN_ZONE:
    for grp_val in GROUPS_FOR_ZONE_COMPARISON:
        grp_label = "TIL-High" if grp_val == "TIL_high" else "TIL-Low"
        pb_by_zone = {zone: pseudobulk(zone, grp_val) for zone in set(z for pair in ZONE_PAIRS for z in pair)}

        for zone_a, zone_b in ZONE_PAIRS:
            pb_a = pb_by_zone[zone_a]
            pb_b = pb_by_zone[zone_b]

            # Only use samples present in BOTH zones (paired design)
            shared_samples = pb_a.index.intersection(pb_b.index)
            if len(shared_samples) < 3:
                print(f"  [SKIP] {zone_a} vs {zone_b} in {grp_label}: "
                      f"only {len(shared_samples)} shared samples")
                continue

            # Paired t-statistic ranking
            rnk = paired_t_rank(pb_a, pb_b, shared_samples)

            label = f"{zone_a}_vs_{zone_b}_in_{grp_label}"
            print(f"\n--- {label} (n={len(shared_samples)} paired samples, "
                  f"ranking=paired t-stat) ---")
            print(f"  Ranking stats: median={rnk.median():.3f}, "
                  f"mean={rnk.mean():.3f}, "
                  f"min={rnk.min():.3f}, max={rnk.max():.3f}, "
                  f"n_zero={(rnk == 0).sum()}")

            res = run_gsea(rnk, label)
            res["zone_A"] = zone_a
            res["zone_B"] = zone_b
            res["group"] = grp_label
            res["n_paired_samples"] = len(shared_samples)
            res["comparison_type"] = "between_zone"
            res["ranking_metric"] = "paired_t"
            print_top(res, zone_a, zone_b)
            all_results.append(res)

print(f"\nGSEA: {time.time() - t0:.0f} s")

## Combined table

In [ ]:
combined_df = pd.concat(all_results, ignore_index=True)
output.save_table(OUT, "4b_gsea_all_comparisons_combined", combined_df, panel="Supp Table 4b", index=False)
print(f"Total comparisons: {combined_df['comparison'].nunique()}")
print(f"Total gene set tests: {len(combined_df)}")

print("\nSignificant (FDR<0.05) per comparison:")
for comp, grp in combined_df.groupby("comparison"):
    sig = grp["FDR"].astype(float) < 0.05
    n_pos = ((grp["NES"].astype(float) > 0) & sig).sum()
    n_neg = ((grp["NES"].astype(float) < 0) & sig).sum()
    print(f"  {comp}: {sig.sum()}/{len(grp)} ({n_pos} pos, {n_neg} neg) [{grp['ranking_metric'].iloc[0]}]")

## Comparison with the published table

In [ ]:
pub = pd.read_csv(PUBLISHED)
new = pd.read_csv(OUT / "4b_gsea_all_comparisons_combined.csv")
print(f"rows: published {len(pub)}, this run {len(new)}; same columns: {list(pub.columns) == list(new.columns)}")
key = ["comparison", "gene_set"]
m = pub.merge(new, on=key, how="outer", suffixes=("_pub", "_new"), indicator=True)
print(m["_merge"].value_counts().to_string())
both = m[m["_merge"] == "both"]
num = ["ES", "NES", "PVAL", "FDR", "FWER p-val"]
diffs = pd.DataFrame({c: (both[f"{c}_pub"] - both[f"{c}_new"]).abs().groupby(both["comparison"]).max() for c in num})
diffs["lead genes identical"] = (both["Lead_genes_pub"] == both["Lead_genes_new"]).groupby(both["comparison"]).mean()
sig_pub = both["FDR_pub"] < 0.05
sig_new = both["FDR_new"] < 0.05
diffs["FDR<0.05 calls differing"] = (sig_pub != sig_new).groupby(both["comparison"]).sum()
print("Maximum absolute differences per comparison:")
print(diffs.to_string())

In [ ]:
print(output.versions(["numpy", "pandas", "gseapy"]).to_string(index=False))